# CHB-MIT protocol audit

This notebook audits CHB-MIT recording coverage, channel compatibility, and seizure-event eligibility. It writes the frozen `protocol_v2` artifacts used by the downstream experiments.

The 4-hour distance rule is used only to define clean interictal windows. Final forecasting-event eligibility is recomputed later in this notebook from preictal coverage, temporal separation from the preceding seizure, and fixed-montage coverage.


In [ ]:

!pip install -q "mne>=1.8,<2"

from google.colab import drive
drive.mount("/content/drive")

from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional
import hashlib
import json
import re

import mne
import numpy as np
import pandas as pd

mne.set_log_level("ERROR")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)


In [ ]:

@dataclass(frozen=True)
class Protocol:
    patients: tuple = ("chb05", "chb06", "chb08", "chb12")
    sop_min: int = 30
    sph_min: int = 5
    postictal_buffer_hours: float = 4.0
    window_sec: float = 10.0
    stride_sec: float = 5.0
    expected_sfreq_hz: float = 256.0
    min_preictal_coverage_fraction: float = 0.95
    random_seeds: tuple = (2026, 2027, 2028, 2029, 2030)

PROTOCOL = Protocol()

DATA_ROOT = Path("/content/drive/MyDrive/chb-mit-scalp-eeg-database-1.0.0")
OUT_ROOT = Path(
    "/content/drive/MyDrive/EEG_Research/"
    "continual_graph_forecasting/protocol_v1"
)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

def canonical_json(obj) -> str:
    return json.dumps(obj, sort_keys=True, separators=(",", ":"))

protocol_dict = asdict(PROTOCOL)
protocol_hash = hashlib.sha256(
    canonical_json(protocol_dict).encode("utf-8")
).hexdigest()

protocol_path = OUT_ROOT / "protocol.json"

if protocol_path.exists():
    existing = json.loads(protocol_path.read_text())
    if existing.get("protocol_hash") != protocol_hash:
        raise RuntimeError(
            "A different protocol already exists in this output folder."
        )
else:
    protocol_path.write_text(json.dumps({
        "protocol_hash": protocol_hash,
        "protocol": protocol_dict,
    }, indent=2))

print("DATA_ROOT :", DATA_ROOT)
print("OUT_ROOT  :", OUT_ROOT)
print("Protocol  :", protocol_hash[:16], "...")
print(json.dumps(protocol_dict, indent=2))


In [ ]:

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"CHB-MIT root not found: {DATA_ROOT}\n"
        "Change DATA_ROOT only if your Drive location is different."
    )

path_rows = []
for patient in PROTOCOL.patients:
    pdir = DATA_ROOT / patient
    summary = pdir / f"{patient}-summary.txt"
    edfs = sorted(pdir.glob("*.edf")) if pdir.exists() else []
    path_rows.append({
        "patient": patient,
        "patient_dir_exists": pdir.exists(),
        "summary_exists": summary.exists(),
        "n_edf_files": len(edfs),
    })

path_audit = pd.DataFrame(path_rows)
display(path_audit)

if not path_audit["patient_dir_exists"].all():
    raise FileNotFoundError("At least one selected patient folder is missing.")
if not path_audit["summary_exists"].all():
    raise FileNotFoundError("At least one selected patient summary file is missing.")
if (path_audit["n_edf_files"] == 0).any():
    raise FileNotFoundError("At least one selected patient contains no EDF files.")


In [ ]:

FILE_BLOCK_RE = re.compile(
    r"File Name:\s*(?P<file>\S+)(?P<body>.*?)(?=\nFile Name:|\Z)",
    flags=re.S,
)

def clock_to_seconds(clock_text: str) -> int:
    # CHB-MIT summaries may contain hours such as 24:21:32.
    h, m, s = map(int, clock_text.strip().split(":"))
    if h < 0 or not (0 <= m < 60) or not (0 <= s < 60):
        raise ValueError(f"Invalid clock time: {clock_text}")
    return h * 3600 + m * 60 + s

def unwrap_clock(raw_seconds: int, reference_seconds: Optional[float]) -> float:
    x = float(raw_seconds)
    if reference_seconds is None:
        return x
    while x < reference_seconds:
        x += 24 * 3600
    return x

def parse_summary(patient_dir: Path, patient_id: str) -> pd.DataFrame:
    summary_path = patient_dir / f"{patient_id}-summary.txt"
    text = summary_path.read_text(errors="replace")

    rows = []
    previous_start_abs = None

    for order, match in enumerate(FILE_BLOCK_RE.finditer(text)):
        fname = match.group("file")
        body = match.group("body")

        start_match = re.search(r"File Start Time:\s*([0-9]+:[0-9]+:[0-9]+)", body)
        end_match = re.search(r"File End Time:\s*([0-9]+:[0-9]+:[0-9]+)", body)
        n_match = re.search(r"Number of Seizures in File:\s*(\d+)", body)

        if not (start_match and end_match and n_match):
            raise ValueError(f"Could not parse required metadata for {patient_id}/{fname}")

        raw_start = clock_to_seconds(start_match.group(1))
        raw_end = clock_to_seconds(end_match.group(1))

        start_abs = unwrap_clock(raw_start, previous_start_abs)
        end_abs = float(raw_end)
        while end_abs <= start_abs:
            end_abs += 24 * 3600

        n_seizures = int(n_match.group(1))
        onsets = [
            int(x) for x in re.findall(
                r"Seizure(?:\s+\d+)?\s+Start Time:\s*(\d+)\s*seconds", body
            )
        ]
        offsets = [
            int(x) for x in re.findall(
                r"Seizure(?:\s+\d+)?\s+End Time:\s*(\d+)\s*seconds", body
            )
        ]

        if not (len(onsets) == len(offsets) == n_seizures):
            raise ValueError(
                f"Seizure-count mismatch in {patient_id}/{fname}: "
                f"declared={n_seizures}, starts={len(onsets)}, ends={len(offsets)}"
            )

        rows.append({
            "patient": patient_id,
            "file_order": order,
            "file": fname,
            "summary_start_clock": start_match.group(1),
            "summary_end_clock": end_match.group(1),
            "summary_start_abs_sec": start_abs,
            "summary_end_abs_sec": end_abs,
            "summary_duration_sec": end_abs - start_abs,
            "n_seizures": n_seizures,
            "seizures_rel_sec": list(zip(onsets, offsets)),
        })
        previous_start_abs = start_abs

    if not rows:
        raise ValueError(f"No EDF file blocks parsed from {summary_path}")

    return pd.DataFrame(rows)

assert clock_to_seconds("24:21:32") == 24 * 3600 + 21 * 60 + 32
assert unwrap_clock(
    clock_to_seconds("00:21:39"),
    clock_to_seconds("23:21:32")
) == 24 * 3600 + 21 * 60 + 39

print("Summary parser unit tests: PASS")


In [ ]:

def canonical_channel_name(name: str) -> Optional[str]:
    x = name.strip().upper()

    if x in {"", "-", "--", "."}:
        return None
    if x.startswith("ECG") or x.startswith("VNS"):
        return None

    # MNE can append -0, -1, ... when EDF channel labels are duplicated.
    x = re.sub(r"-(\d+)$", "", x)
    return x

def audit_patient(patient: str) -> pd.DataFrame:
    patient_dir = DATA_ROOT / patient
    summary_df = parse_summary(patient_dir, patient)

    rows = []
    for _, rec in summary_df.iterrows():
        edf_path = patient_dir / rec["file"]
        if not edf_path.exists():
            raise FileNotFoundError(f"Missing EDF referenced by summary: {edf_path}")

        raw = mne.io.read_raw_edf(edf_path, preload=False, verbose="ERROR")

        sfreq = float(raw.info["sfreq"])
        n_times = int(raw.n_times)
        duration_sec = n_times / sfreq

        raw_names = list(raw.ch_names)
        canonical = [
            c for c in (canonical_channel_name(x) for x in raw_names)
            if c is not None
        ]

        rows.append({
            "patient": patient,
            "file_order": int(rec["file_order"]),
            "file": rec["file"],
            "summary_start_abs_sec": float(rec["summary_start_abs_sec"]),
            "summary_end_abs_sec": float(rec["summary_end_abs_sec"]),
            "summary_duration_sec": float(rec["summary_duration_sec"]),
            "edf_sfreq_hz": sfreq,
            "edf_n_times": n_times,
            "edf_duration_sec": duration_sec,
            "duration_delta_sec": duration_sec - float(rec["summary_duration_sec"]),
            "n_raw_channels": len(raw_names),
            "n_eeg_like_channels": len(canonical),
            "raw_channels": raw_names,
            "canonical_channels": canonical,
            "n_seizures": int(rec["n_seizures"]),
            "seizures_rel_sec": rec["seizures_rel_sec"],
        })
        raw.close()

    out = pd.DataFrame(rows).sort_values("file_order").reset_index(drop=True)

    # Use the EDF header duration to represent observed signal coverage.
    out["edf_start_abs_sec"] = out["summary_start_abs_sec"]
    out["edf_end_abs_sec"] = out["edf_start_abs_sec"] + out["edf_duration_sec"]

    out["gap_from_previous_sec"] = np.nan
    for i in range(1, len(out)):
        out.loc[i, "gap_from_previous_sec"] = (
            out.loc[i, "edf_start_abs_sec"] - out.loc[i - 1, "edf_end_abs_sec"]
        )

    return out

file_manifests = []
for patient in PROTOCOL.patients:
    print(f"Auditing {patient} ...")
    file_manifests.append(audit_patient(patient))

files_df = pd.concat(file_manifests, ignore_index=True)

bad_sfreq = files_df[
    ~np.isclose(
        files_df["edf_sfreq_hz"],
        PROTOCOL.expected_sfreq_hz,
        rtol=0,
        atol=1e-9,
    )
]
if len(bad_sfreq):
    display(bad_sfreq[["patient", "file", "edf_sfreq_hz"]])
    raise RuntimeError("Unexpected sampling rate found. Stop before preprocessing.")

large_duration_delta = files_df[files_df["duration_delta_sec"].abs() > 1.0]

print("\nFiles audited:", len(files_df))
print("Sampling rates:", sorted(files_df["edf_sfreq_hz"].unique()))
print("Files with |EDF duration - summary duration| > 1 s:", len(large_duration_delta))

gap_summary = (
    files_df.dropna(subset=["gap_from_previous_sec"])
    .groupby("patient")["gap_from_previous_sec"]
    .agg(["count", "min", "median", "max"])
)

print("\nObserved file-to-file gaps (seconds):")
display(gap_summary)

if (files_df["gap_from_previous_sec"].dropna() < -1.0).any():
    print("WARNING: overlapping EDF time intervals >1 s detected; inspect before modeling.")

files_df.to_pickle(OUT_ROOT / "file_manifest.pkl")
files_df.drop(
    columns=["raw_channels", "canonical_channels", "seizures_rel_sec"]
).to_csv(OUT_ROOT / "file_manifest.csv", index=False)

print("Saved file manifest.")


In [ ]:

all_channel_sets = [set(chs) for chs in files_df["canonical_channels"]]
common_channels = sorted(set.intersection(*all_channel_sets))
union_channels = sorted(set.union(*all_channel_sets))

channel_rows = []
for patient in PROTOCOL.patients:
    patient_sets = [
        set(x) for x in files_df.loc[
            files_df["patient"] == patient, "canonical_channels"
        ]
    ]
    patient_common = sorted(set.intersection(*patient_sets))
    patient_union = sorted(set.union(*patient_sets))

    channel_rows.append({
        "patient": patient,
        "common_within_patient": len(patient_common),
        "union_within_patient": len(patient_union),
        "common_channels": patient_common,
    })

channel_audit = pd.DataFrame(channel_rows)

print("Channels common to EVERY selected EDF file:", len(common_channels))
print(common_channels)
print("\nPer-patient channel consistency:")
display(channel_audit[["patient", "common_within_patient", "union_within_patient"]])

(OUT_ROOT / "common_channels_all_selected_files.json").write_text(
    json.dumps(common_channels, indent=2)
)


## Event coverage audit

The next cell creates the seizure-event table used for the audit. Its provisional `eligible` field is retained only for comparison with the corrected rule. The final paper eligibility is defined by `protocol_v2` below.


In [ ]:

def interval_union_length(intervals, start, end) -> float:
    # Compute observed recording time inside [start, end), without double counting.
    clipped = []
    for a, b in intervals:
        left = max(float(a), float(start))
        right = min(float(b), float(end))
        if right > left:
            clipped.append((left, right))

    if not clipped:
        return 0.0

    clipped.sort()
    merged = [list(clipped[0])]

    for a, b in clipped[1:]:
        if a <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], b)
        else:
            merged.append([a, b])

    return float(sum(b - a for a, b in merged))

def build_seizure_manifest(patient: str, patient_files: pd.DataFrame) -> pd.DataFrame:
    patient_files = patient_files.sort_values("file_order").reset_index(drop=True)
    recording_intervals = list(zip(
        patient_files["edf_start_abs_sec"],
        patient_files["edf_end_abs_sec"],
    ))

    events = []

    for _, row in patient_files.iterrows():
        for seizure_idx_in_file, pair in enumerate(row["seizures_rel_sec"], start=1):
            on_rel, off_rel = pair
            onset = float(row["edf_start_abs_sec"]) + float(on_rel)
            offset = float(row["edf_start_abs_sec"]) + float(off_rel)

            if not (row["edf_start_abs_sec"] <= onset < row["edf_end_abs_sec"]):
                raise RuntimeError(f"Seizure onset outside EDF interval: {patient}/{row['file']}")
            if not (onset < offset <= row["edf_end_abs_sec"] + 1.0):
                raise RuntimeError(f"Seizure offset outside EDF interval: {patient}/{row['file']}")

            events.append({
                "patient": patient,
                "file": row["file"],
                "seizure_idx_in_file": seizure_idx_in_file,
                "onset_abs_sec": onset,
                "offset_abs_sec": offset,
                "onset_rel_sec": int(on_rel),
                "offset_rel_sec": int(off_rel),
            })

    events = sorted(events, key=lambda x: x["onset_abs_sec"])

    sop_sec = PROTOCOL.sop_min * 60.0
    sph_sec = PROTOCOL.sph_min * 60.0
    post_buffer_sec = PROTOCOL.postictal_buffer_hours * 3600.0

    previous_offset = None

    for event_number, event in enumerate(events, start=1):
        onset = event["onset_abs_sec"]
        pre_start = onset - (sop_sec + sph_sec)
        pre_end = onset - sph_sec

        coverage_sec = interval_union_length(
            recording_intervals, pre_start, pre_end
        )
        coverage_fraction = coverage_sec / sop_sec

        if previous_offset is None:
            seconds_from_prev_end_to_preictal = np.nan
            clean_from_previous = True
        else:
            seconds_from_prev_end_to_preictal = pre_start - previous_offset
            clean_from_previous = (
                seconds_from_prev_end_to_preictal >= post_buffer_sec
            )

        sufficient_coverage = (
            coverage_fraction >= PROTOCOL.min_preictal_coverage_fraction
        )

        reasons = []
        if not sufficient_coverage:
            reasons.append(
                f"preictal_coverage<{PROTOCOL.min_preictal_coverage_fraction:.2f}"
            )
        if not clean_from_previous:
            reasons.append(
                f"previous_seizure_within_{PROTOCOL.postictal_buffer_hours:g}h_buffer"
            )

        event.update({
            "event_number": event_number,
            "preictal_start_abs_sec": pre_start,
            "preictal_end_abs_sec": pre_end,
            "preictal_coverage_sec": coverage_sec,
            "preictal_coverage_fraction": coverage_fraction,
            "seconds_from_prev_end_to_preictal": seconds_from_prev_end_to_preictal,
            "clean_from_previous_seizure": bool(clean_from_previous),
            "sufficient_preictal_coverage": bool(sufficient_coverage),
            "eligible": bool(clean_from_previous and sufficient_coverage),
            "exclusion_reason": ";".join(reasons),
        })

        previous_offset = event["offset_abs_sec"]

    return pd.DataFrame(events)

seizure_manifests = []
for patient in PROTOCOL.patients:
    pfiles = files_df[files_df["patient"] == patient].copy()
    seizure_manifests.append(build_seizure_manifest(patient, pfiles))

seizures_df = pd.concat(seizure_manifests, ignore_index=True)
seizures_df["event_id"] = (
    seizures_df["patient"]
    + "_E"
    + seizures_df.groupby("patient").cumcount().add(1).astype(str).str.zfill(2)
)

summary = (
    seizures_df.groupby("patient")
    .agg(
        raw_seizures=("event_id", "count"),
        eligible_seizures=("eligible", "sum"),
        min_preictal_coverage=("preictal_coverage_fraction", "min"),
        median_preictal_coverage=("preictal_coverage_fraction", "median"),
    )
    .reset_index()
)

display(summary)

print("\nEvent-level audit:")
display(
    seizures_df[
        [
            "event_id",
            "file",
            "onset_rel_sec",
            "preictal_coverage_fraction",
            "clean_from_previous_seizure",
            "eligible",
            "exclusion_reason",
        ]
    ]
)

seizures_df.to_csv(OUT_ROOT / "seizure_manifest.csv", index=False)
summary.to_csv(OUT_ROOT / "patient_event_audit.csv", index=False)

print("Saved seizure manifest and patient event audit.")


In [ ]:
# ============================================================
# Montage audit and corrected forecasting-event rule
# ============================================================
# The 4-hour rule is retained only for clean interictal sampling.
# Forecasting-event eligibility uses preictal coverage and requires
# the preictal interval to begin after the previous seizure ends.

from collections import defaultdict
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Check channel stability for chb05/chb06/chb08
# ------------------------------------------------------------

stable_pilot_patients = ("chb05", "chb06", "chb08")

stable_channel_sets = []

for patient in stable_pilot_patients:
    patient_rows = files_df.loc[
        files_df["patient"] == patient,
        "canonical_channels"
    ]

    for channels in patient_rows:
        stable_channel_sets.append(set(channels))

common_stable_channels = sorted(
    set.intersection(*stable_channel_sets)
)

print("=" * 80)
print("A. CHANNELS COMMON TO chb05 + chb06 + chb08")
print("=" * 80)

print("Number of common channels:", len(common_stable_channels))
print(common_stable_channels)


# ------------------------------------------------------------
# 2. Diagnose chb12 montage changes
# ------------------------------------------------------------

chb12_files = files_df[
    files_df["patient"] == "chb12"
].copy()

# Exact channel-set signature for each EDF.
chb12_files["montage_signature"] = chb12_files[
    "canonical_channels"
].apply(
    lambda x: tuple(sorted(set(x)))
)

unique_signatures = list(
    chb12_files["montage_signature"].drop_duplicates()
)

signature_to_id = {
    sig: f"M{i+1:02d}"
    for i, sig in enumerate(unique_signatures)
}

chb12_files["montage_id"] = chb12_files[
    "montage_signature"
].map(signature_to_id)

montage_summary = (
    chb12_files
    .groupby("montage_id")
    .agg(
        n_files=("file", "count"),
        n_channels=("montage_signature", lambda x: len(x.iloc[0])),
        files=("file", lambda x: ", ".join(x))
    )
    .reset_index()
)

print("\n" + "=" * 80)
print("B. chb12 MONTAGE VARIANTS")
print("=" * 80)

display(montage_summary)

print("\nDetailed channel sets:")

for _, row in montage_summary.iterrows():
    montage_id = row["montage_id"]

    sig = chb12_files.loc[
        chb12_files["montage_id"] == montage_id,
        "montage_signature"
    ].iloc[0]

    print(f"\n{montage_id}: {len(sig)} channels")
    print(list(sig))


# ------------------------------------------------------------
# 3. Attach montage IDs to chb12 seizures
# ------------------------------------------------------------

chb12_events = seizures_df[
    seizures_df["patient"] == "chb12"
].copy()

chb12_events = chb12_events.merge(
    chb12_files[["file", "montage_id"]],
    on="file",
    how="left"
)

print("\n" + "=" * 80)
print("C. chb12 SEIZURES BY MONTAGE")
print("=" * 80)

display(
    chb12_events[
        [
            "event_id",
            "file",
            "montage_id",
            "onset_rel_sec",
            "preictal_coverage_fraction",
            "eligible",
            "exclusion_reason"
        ]
    ]
)


# ------------------------------------------------------------
# 4. Recompute candidate seizure eligibility correctly
# ------------------------------------------------------------
#
# We REMOVE the 4-hour requirement from seizure eligibility.
#
# A seizure is a candidate forecasting event when:
#
# 1. >=95% of its SOP is actually recorded.
# 2. Its preictal interval does not overlap the previous seizure.
#
# The 4-hour rule will still be retained later for selecting
# CLEAN INTERICTAL negative training examples.
# ------------------------------------------------------------

candidate_parts = []

for patient in PROTOCOL.patients:

    pevents = (
        seizures_df[
            seizures_df["patient"] == patient
        ]
        .sort_values("onset_abs_sec")
        .copy()
    )

    previous_offset = None

    clear_flags = []
    separation_seconds = []

    for _, event in pevents.iterrows():

        if previous_offset is None:
            clear = True
            separation = np.nan

        else:
            # Time from previous seizure END to beginning of the
            # current event's preictal interval.
            separation = (
                float(event["preictal_start_abs_sec"])
                - float(previous_offset)
            )

            # >= 0 means the defined preictal interval itself
            # does not overlap the previous seizure.
            clear = separation >= 0.0

        clear_flags.append(bool(clear))
        separation_seconds.append(separation)

        previous_offset = float(event["offset_abs_sec"])

    pevents["preictal_clear_of_previous_seizure"] = clear_flags

    pevents[
        "seconds_prev_seizure_end_to_preictal_start"
    ] = separation_seconds

    pevents["candidate_eligible_v2"] = (
        pevents["sufficient_preictal_coverage"]
        & pevents["preictal_clear_of_previous_seizure"]
    )

    reasons = []

    for _, event in pevents.iterrows():

        r = []

        if not bool(event["sufficient_preictal_coverage"]):
            r.append(
                f"preictal_coverage<"
                f"{PROTOCOL.min_preictal_coverage_fraction:.2f}"
            )

        if not bool(event["preictal_clear_of_previous_seizure"]):
            r.append("preictal_overlaps_previous_seizure")

        reasons.append(";".join(r))

    pevents["candidate_exclusion_reason_v2"] = reasons

    candidate_parts.append(pevents)


candidate_events = pd.concat(
    candidate_parts,
    ignore_index=True
)


# ------------------------------------------------------------
# 5. Compare old and corrected eligibility
# ------------------------------------------------------------

comparison = (
    candidate_events
    .groupby("patient")
    .agg(
        raw_seizures=("event_id", "count"),
        eligible_v1=("eligible", "sum"),
        candidate_eligible_v2=("candidate_eligible_v2", "sum")
    )
    .reset_index()
)

print("\n" + "=" * 80)
print("D. OLD vs CORRECTED CANDIDATE ELIGIBILITY")
print("=" * 80)

display(comparison)


# ------------------------------------------------------------
# 6. Show exactly which events change
# ------------------------------------------------------------

changed = candidate_events[
    candidate_events["eligible"]
    != candidate_events["candidate_eligible_v2"]
].copy()

print("\n" + "=" * 80)
print("E. EVENTS WHOSE ELIGIBILITY CHANGES")
print("=" * 80)

display(
    changed[
        [
            "event_id",
            "file",
            "preictal_coverage_fraction",
            "seconds_prev_seizure_end_to_preictal_start",
            "eligible",
            "candidate_eligible_v2",
            "exclusion_reason",
            "candidate_exclusion_reason_v2"
        ]
    ]
)


# ------------------------------------------------------------
# 7. chb12 candidate events after corrected rule
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("F. chb12 CANDIDATE EVENTS UNDER CORRECTED RULE")
print("=" * 80)

display(
    candidate_events.loc[
        candidate_events["patient"] == "chb12",
        [
            "event_id",
            "file",
            "preictal_coverage_fraction",
            "seconds_prev_seizure_end_to_preictal_start",
            "candidate_eligible_v2",
            "candidate_exclusion_reason_v2"
        ]
    ]
)

print("\nDIAGNOSTIC CELL COMPLETE.")
print("Nothing has been overwritten.")


In [ ]:
# ============================================================
# Protocol v2: final event and montage eligibility
# ============================================================
# Event eligibility requires:
#   1. at least 95% preictal recording coverage;
#   2. no overlap between the preictal interval and the previous seizure;
#   3. a compatible seizure-onset EDF; and
#   4. at least 95% preictal coverage using the fixed 22-channel montage.
# The 4-hour rule is used only for clean interictal negatives.
#
# Changes from protocol_v1:
#
# 1. The 4-hour seizure distance is NOT an event-eligibility rule.
#    It remains only the rule for selecting clean interictal negatives.
#
# 2. A forecasting event is eligible when:
#       a) >=95% of the 30-min SOP is recorded
#       b) its preictal interval does not overlap the previous seizure
#       c) >=95% of its preictal interval is recorded using the fixed
#          compatible 22-channel montage
#
# 3. All models use exactly the same 22 bipolar EEG channels.
#
# 4. Files with incompatible montages are excluded from modeling.
#
# Nothing in protocol_v1 is overwritten.
# ============================================================

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Fixed EEG montage
# ------------------------------------------------------------

FIXED_CHANNELS = tuple(sorted(common_stable_channels))

assert len(FIXED_CHANNELS) == 22, (
    f"Expected 22 fixed channels, found {len(FIXED_CHANNELS)}"
)

print("=" * 80)
print("FIXED EEG MONTAGE")
print("=" * 80)

print("Number of channels:", len(FIXED_CHANNELS))

for i, ch in enumerate(FIXED_CHANNELS, start=1):
    print(f"{i:02d}. {ch}")


# ------------------------------------------------------------
# 2. Mark EDF files compatible with the fixed montage
# ------------------------------------------------------------

files_v2 = files_df.copy()

fixed_set = set(FIXED_CHANNELS)

files_v2["montage_compatible"] = files_v2[
    "canonical_channels"
].apply(
    lambda channels: fixed_set.issubset(set(channels))
)

compatibility_summary = (
    files_v2
    .groupby("patient")
    .agg(
        total_files=("file", "count"),
        compatible_files=("montage_compatible", "sum")
    )
    .reset_index()
)

compatibility_summary["excluded_files"] = (
    compatibility_summary["total_files"]
    - compatibility_summary["compatible_files"]
)

print("\n" + "=" * 80)
print("FILE MONTAGE COMPATIBILITY")
print("=" * 80)

display(compatibility_summary)


print("\nExcluded files:")

excluded_files = files_v2[
    ~files_v2["montage_compatible"]
][
    ["patient", "file", "n_raw_channels"]
].copy()

display(excluded_files)


# ------------------------------------------------------------
# 3. Compatible recording intervals
# ------------------------------------------------------------

compatible_intervals = {}

for patient in PROTOCOL.patients:

    tmp = files_v2[
        (files_v2["patient"] == patient)
        & (files_v2["montage_compatible"])
    ]

    compatible_intervals[patient] = list(
        zip(
            tmp["edf_start_abs_sec"].astype(float),
            tmp["edf_end_abs_sec"].astype(float)
        )
    )


# ------------------------------------------------------------
# 4. Final event eligibility
# ------------------------------------------------------------

events_v2 = candidate_events.copy()

# Determine whether the EDF containing seizure onset itself
# has the required fixed montage.

file_compatibility_map = (
    files_v2
    .set_index(["patient", "file"])["montage_compatible"]
    .to_dict()
)

events_v2["onset_file_montage_compatible"] = events_v2.apply(
    lambda r: bool(
        file_compatibility_map.get(
            (r["patient"], r["file"]),
            False
        )
    ),
    axis=1
)


# Measure how much of each 30-min SOP is actually available
# WITH THE FIXED MONTAGE.

compatible_coverage_sec = []
compatible_coverage_fraction = []

sop_sec = PROTOCOL.sop_min * 60.0

for _, event in events_v2.iterrows():

    intervals = compatible_intervals[event["patient"]]

    observed = interval_union_length(
        intervals,
        float(event["preictal_start_abs_sec"]),
        float(event["preictal_end_abs_sec"])
    )

    compatible_coverage_sec.append(observed)
    compatible_coverage_fraction.append(
        observed / sop_sec
    )

events_v2[
    "compatible_preictal_coverage_sec"
] = compatible_coverage_sec

events_v2[
    "compatible_preictal_coverage_fraction"
] = compatible_coverage_fraction

events_v2[
    "sufficient_compatible_preictal_coverage"
] = (
    events_v2[
        "compatible_preictal_coverage_fraction"
    ]
    >= PROTOCOL.min_preictal_coverage_fraction
)


# Final eligibility:
#
# - original coverage sufficient
# - no overlap with previous seizure
# - onset recording uses fixed montage
# - >=95% preictal coverage exists in fixed montage

events_v2["eligible_v2"] = (
    events_v2["sufficient_preictal_coverage"]
    & events_v2["preictal_clear_of_previous_seizure"]
    & events_v2["onset_file_montage_compatible"]
    & events_v2[
        "sufficient_compatible_preictal_coverage"
    ]
)


# ------------------------------------------------------------
# 5. Explicit exclusion reasons
# ------------------------------------------------------------

final_reasons = []

for _, event in events_v2.iterrows():

    reasons = []

    if not bool(event["sufficient_preictal_coverage"]):
        reasons.append(
            f"preictal_coverage<"
            f"{PROTOCOL.min_preictal_coverage_fraction:.2f}"
        )

    if not bool(
        event["preictal_clear_of_previous_seizure"]
    ):
        reasons.append(
            "preictal_overlaps_previous_seizure"
        )

    if not bool(
        event["onset_file_montage_compatible"]
    ):
        reasons.append(
            "incompatible_seizure_file_montage"
        )

    if not bool(
        event[
            "sufficient_compatible_preictal_coverage"
        ]
    ):
        reasons.append(
            "insufficient_fixed_montage_preictal_coverage"
        )

    final_reasons.append(";".join(reasons))

events_v2["exclusion_reason_v2"] = final_reasons


# ------------------------------------------------------------
# 6. Final event summary
# ------------------------------------------------------------

event_summary_v2 = (
    events_v2
    .groupby("patient")
    .agg(
        raw_seizures=("event_id", "count"),
        eligible_events=("eligible_v2", "sum"),
        median_compatible_coverage=(
            "compatible_preictal_coverage_fraction",
            "median"
        )
    )
    .reset_index()
)

print("\n" + "=" * 80)
print("FINAL PROTOCOL V2 EVENT SUMMARY")
print("=" * 80)

display(event_summary_v2)


print("\nEligible events:")

display(
    events_v2.loc[
        events_v2["eligible_v2"],
        [
            "patient",
            "event_id",
            "file",
            "preictal_coverage_fraction",
            "compatible_preictal_coverage_fraction"
        ]
    ]
)


print("\nExcluded chb12 events:")

display(
    events_v2.loc[
        (events_v2["patient"] == "chb12")
        & (~events_v2["eligible_v2"]),
        [
            "event_id",
            "file",
            "preictal_coverage_fraction",
            "compatible_preictal_coverage_fraction",
            "exclusion_reason_v2"
        ]
    ]
)


# ------------------------------------------------------------
# 7. Create protocol_v2 directory
# ------------------------------------------------------------

OUT_ROOT_V2 = Path(
    "/content/drive/MyDrive/EEG_Research/"
    "continual_graph_forecasting/protocol_v2"
)

OUT_ROOT_V2.mkdir(
    parents=True,
    exist_ok=True
)


protocol_v2 = {

    "study": (
        "Continual Graph Learning for Personalized EEG "
        "Seizure Forecasting Under Temporal Distribution Shift"
    ),

    "dataset": "CHB-MIT",

    "patients": list(PROTOCOL.patients),

    "forecasting": {
        "sop_min": PROTOCOL.sop_min,
        "sph_min": PROTOCOL.sph_min,
        "preictal_definition": (
            "[seizure_onset-(SOP+SPH), "
            "seizure_onset-SPH)"
        )
    },

    "windowing": {
        "window_sec": PROTOCOL.window_sec,
        "stride_sec": PROTOCOL.stride_sec,
        "windows_cross_edf_boundary": False
    },

    "event_eligibility": {
        "min_preictal_coverage_fraction":
            PROTOCOL.min_preictal_coverage_fraction,

        "previous_seizure_rule":
            "preictal interval must not overlap previous seizure",

        "requires_fixed_montage": True
    },

    "interictal_sampling": {
        "minimum_distance_from_any_seizure_hours":
            PROTOCOL.postictal_buffer_hours
    },

    "montage": {
        "type": "fixed bipolar channel montage",
        "n_channels": len(FIXED_CHANNELS),
        "channels": list(FIXED_CHANNELS)
    },

    "random_seeds": list(PROTOCOL.random_seeds),

    "source_protocol_v1_hash":
        protocol_hash
}


def canonical_json_v2(obj):
    return json.dumps(
        obj,
        sort_keys=True,
        separators=(",", ":")
    )


protocol_v2_hash = hashlib.sha256(
    canonical_json_v2(protocol_v2).encode("utf-8")
).hexdigest()


protocol_v2_payload = {
    "protocol_hash": protocol_v2_hash,
    "protocol": protocol_v2
}


protocol_v2_path = (
    OUT_ROOT_V2 / "protocol.json"
)


if protocol_v2_path.exists():

    existing = json.loads(
        protocol_v2_path.read_text()
    )

    if (
        existing.get("protocol_hash")
        != protocol_v2_hash
    ):
        raise RuntimeError(
            "protocol_v2 already exists with different "
            "settings. Do not silently overwrite it."
        )

else:

    protocol_v2_path.write_text(
        json.dumps(
            protocol_v2_payload,
            indent=2
        )
    )


# ------------------------------------------------------------
# 8. Save final file and event manifests
# ------------------------------------------------------------

events_v2.to_csv(
    OUT_ROOT_V2 / "seizure_manifest_v2.csv",
    index=False
)

event_summary_v2.to_csv(
    OUT_ROOT_V2 / "patient_event_summary_v2.csv",
    index=False
)


# CSV-friendly file manifest
files_v2[
    [
        "patient",
        "file_order",
        "file",
        "edf_sfreq_hz",
        "edf_duration_sec",
        "edf_start_abs_sec",
        "edf_end_abs_sec",
        "gap_from_previous_sec",
        "montage_compatible"
    ]
].to_csv(
    OUT_ROOT_V2 / "file_manifest_v2.csv",
    index=False
)


with open(
    OUT_ROOT_V2 / "fixed_channels.json",
    "w"
) as f:

    json.dump(
        list(FIXED_CHANNELS),
        f,
        indent=2
    )


# ------------------------------------------------------------
# 9. Rebuild window manifest using FINAL eligibility
# ------------------------------------------------------------

def build_window_metadata_v2(
    patient,
    patient_files,
    patient_events
):

    rows = []

    window_sec = PROTOCOL.window_sec
    stride_sec = PROTOCOL.stride_sec

    sph_sec = (
        PROTOCOL.sph_min * 60.0
    )

    clean_margin_sec = (
        PROTOCOL.postictal_buffer_hours
        * 3600.0
    )

    # Only compatible EDFs enter modeling.
    usable_files = patient_files[
        patient_files["montage_compatible"]
    ].copy()

    eligible_events = patient_events[
        patient_events["eligible_v2"]
    ].copy()


    for _, frow in usable_files.sort_values(
        "file_order"
    ).iterrows():

        duration = float(
            frow["edf_duration_sec"]
        )

        if duration < window_sec:
            continue


        rel_starts = np.arange(
            0.0,
            duration - window_sec + 1e-9,
            stride_sec,
            dtype=float
        )

        abs_starts = (
            float(frow["edf_start_abs_sec"])
            + rel_starts
        )

        abs_ends = (
            abs_starts + window_sec
        )


        zone = np.full(
            len(rel_starts),
            "background",
            dtype=object
        )

        event_ids = np.full(
            len(rel_starts),
            "",
            dtype=object
        )


        # ------------------------------------------
        # Ictal exclusion
        # ------------------------------------------

        for _, ev in patient_events.iterrows():

            mask = overlaps(
                abs_starts,
                abs_ends,
                float(ev["onset_abs_sec"]),
                float(ev["offset_abs_sec"])
            )

            zone[mask] = "ictal"


        # ------------------------------------------
        # SPH exclusion
        # ------------------------------------------

        for _, ev in patient_events.iterrows():

            mask = (
                (zone == "background")
                & overlaps(
                    abs_starts,
                    abs_ends,
                    float(ev["onset_abs_sec"])
                    - sph_sec,
                    float(ev["onset_abs_sec"])
                )
            )

            zone[mask] = "sph_gap"


        # ------------------------------------------
        # Eligible preictal windows
        # ------------------------------------------

        for _, ev in eligible_events.iterrows():

            mask = (
                (zone == "background")
                & (
                    abs_starts
                    >= float(
                        ev[
                            "preictal_start_abs_sec"
                        ]
                    )
                )
                & (
                    abs_ends
                    <= float(
                        ev[
                            "preictal_end_abs_sec"
                        ]
                    )
                )
            )

            zone[mask] = "preictal"

            event_ids[mask] = (
                ev["event_id"]
            )


        # ------------------------------------------
        # Clean interictal negatives
        #
        # Must be >=4h away from ANY seizure,
        # including seizures excluded from the
        # forecasting-event analysis.
        # ------------------------------------------

        peri_mask = np.zeros(
            len(rel_starts),
            dtype=bool
        )

        for _, ev in patient_events.iterrows():

            peri_mask |= overlaps(
                abs_starts,
                abs_ends,
                float(ev["onset_abs_sec"])
                - clean_margin_sec,
                float(ev["offset_abs_sec"])
                + clean_margin_sec
            )


        clean_mask = (
            (zone == "background")
            & (~peri_mask)
        )

        zone[
            clean_mask
        ] = "clean_interictal"


        remaining_peri = (
            (zone == "background")
            & peri_mask
        )

        zone[
            remaining_peri
        ] = "peri_ictal_exclusion"


        part = pd.DataFrame({

            "patient": patient,

            "file": frow["file"],

            "file_order":
                int(frow["file_order"]),

            "window_index_in_file":
                np.arange(
                    len(rel_starts),
                    dtype=int
                ),

            "start_rel_sec":
                rel_starts,

            "end_rel_sec":
                rel_starts + window_sec,

            "start_abs_sec":
                abs_starts,

            "end_abs_sec":
                abs_ends,

            "zone":
                zone,

            "event_id":
                event_ids

        })

        rows.append(part)


    if not rows:
        return pd.DataFrame()

    return pd.concat(
        rows,
        ignore_index=True
    )


window_parts_v2 = []

for patient in PROTOCOL.patients:

    print(
        f"Building protocol_v2 windows "
        f"for {patient} ..."
    )

    pfiles = files_v2[
        files_v2["patient"] == patient
    ].copy()

    pevents = events_v2[
        events_v2["patient"] == patient
    ].copy()

    window_parts_v2.append(
        build_window_metadata_v2(
            patient,
            pfiles,
            pevents
        )
    )


windows_v2 = pd.concat(
    window_parts_v2,
    ignore_index=True
)


# ------------------------------------------------------------
# 10. Sanity checks
# ------------------------------------------------------------

assert not (
    (
        windows_v2["zone"]
        == "preictal"
    )
    & (
        windows_v2["event_id"]
        == ""
    )
).any()


# Every preictal event ID must correspond to an eligible event.

eligible_ids = set(
    events_v2.loc[
        events_v2["eligible_v2"],
        "event_id"
    ]
)

window_preictal_ids = set(
    windows_v2.loc[
        windows_v2["zone"] == "preictal",
        "event_id"
    ]
)

assert window_preictal_ids.issubset(
    eligible_ids
)


# No incompatible EDF may appear.

compatible_file_keys = set(
    zip(
        files_v2.loc[
            files_v2["montage_compatible"],
            "patient"
        ],
        files_v2.loc[
            files_v2["montage_compatible"],
            "file"
        ]
    )
)

window_file_keys = set(
    zip(
        windows_v2["patient"],
        windows_v2["file"]
    )
)

assert window_file_keys.issubset(
    compatible_file_keys
)


# ------------------------------------------------------------
# 11. Save window manifests
# ------------------------------------------------------------

windows_v2.to_pickle(
    OUT_ROOT_V2
    / "window_manifest_v2.pkl"
)

windows_v2.to_csv(
    OUT_ROOT_V2
    / "window_manifest_v2.csv",
    index=False
)


window_counts_v2 = (
    windows_v2
    .groupby(
        ["patient", "zone"]
    )
    .size()
    .rename("n_windows")
    .reset_index()
)

window_counts_v2.to_csv(
    OUT_ROOT_V2
    / "window_counts_v2.csv",
    index=False
)


# Number of preictal windows per eligible event

preictal_event_counts = (
    windows_v2[
        windows_v2["zone"]
        == "preictal"
    ]
    .groupby(
        ["patient", "event_id"]
    )
    .size()
    .rename("n_preictal_windows")
    .reset_index()
)


# ------------------------------------------------------------
# 12. Final report
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PROTOCOL V2 FROZEN")
print("=" * 80)

print(
    "Protocol hash:",
    protocol_v2_hash
)

print(
    "\nFixed montage:",
    len(FIXED_CHANNELS),
    "channels"
)

print("\nFinal event counts:")
display(
    event_summary_v2
)

print("\nFinal window counts:")
display(
    window_counts_v2
)

print(
    "\nPreictal windows per eligible event:"
)

display(
    preictal_event_counts
)

print(
    "\nOutput directory:",
    OUT_ROOT_V2
)

print("\nSaved files:")

for p in sorted(
    OUT_ROOT_V2.iterdir()
):
    print(" -", p.name)
